In [22]:
# Step 1: Install and Load the Dataset

!pip install datasets

from datasets import load_dataset
import pandas as pd
import re
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

# Load dataset
raw_dataset = load_dataset("wangrongsheng/ag_news")

# Convert to DataFrame
df = pd.DataFrame(raw_dataset['train'])

print(df.head())

                                                text  label
0  Wall St. Bears Claw Back Into the Black (Reute...      2
1  Carlyle Looks Toward Commercial Aerospace (Reu...      2
2  Oil and Economy Cloud Stocks' Outlook (Reuters...      2
3  Iraq Halts Oil Exports from Main Southern Pipe...      2
4  Oil prices soar to all-time record, posing new...      2


In [23]:
# Step 2: Clean and Normalize the Text
def clean_text(text):
    text = text.lower()                         # Convert to lowercase
    text = re.sub(r'[^a-zA-Z\s]', '', text)     # Remove numbers and special characters
    return text

df['text'] = df['text'].apply(clean_text)

print(df['text'].head())

0    wall st bears claw back into the black reuters...
1    carlyle looks toward commercial aerospace reut...
2    oil and economy cloud stocks outlook reuters r...
3    iraq halts oil exports from main southern pipe...
4    oil prices soar to alltime record posing new m...
Name: text, dtype: object


In [24]:
# Step 3: Tokenize the Text
max_words = 10000

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(df['text'])

sequences = tokenizer.texts_to_sequences(df['text'])

print(sequences[:2])

[[392, 325, 1526, 1, 100, 55, 2, 813, 24, 24, 1, 392, 1989, 1, 5, 1, 35, 3894, 738, 296], [1, 1005, 802, 1244, 4133, 24, 24, 876, 746, 303, 1, 1, 22, 4, 4316, 9, 524, 1, 7, 1, 2042, 6, 2, 481, 227, 22, 3766, 1, 6381, 8, 183, 328, 5, 2, 113]]


In [25]:
# Step 4: Pad the Sequences
max_length = 50

X = pad_sequences(
    sequences,
    maxlen=max_length,
    padding='post',
    truncating='post'
)

print(X.shape)

(120000, 50)


In [26]:
# Step 5: One-Hot Encode Labels
y = to_categorical(df['label'])

print(y[:5])

[[0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]]


In [27]:
# Step 6: Split the Dataset
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

print(X_train.shape)
print(X_test.shape)

# Build the RNN Model
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(input_dim=max_words, output_dim=64),
    tf.keras.layers.SimpleRNN(64),
    tf.keras.layers.Dense(4, activation='softmax')
])

model.summary()

(96000, 50)
(24000, 50)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_2 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [28]:
# Step 7: Compile the Model
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
# Train the Model
history = model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/5
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 40s 29ms/step - accuracy: 0.7956 - loss: 0.5810 - val_accuracy: 0.8659 - val_loss: 0.4056
Epoch 2/5
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 29s 25ms/step - accuracy: 0.8809 - loss: 0.3658 - val_accuracy: 0.8594 - val_loss: 0.4142
Epoch 3/5
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 29s 24ms/step - accuracy: 0.8996 - loss: 0.3110 - val_accuracy: 0.8648 - val_loss: 0.3951
Epoch 4/5
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 40s 24ms/step - accuracy: 0.9070 - loss: 0.2812 - val_accuracy: 0.8370 - val_loss: 0.4668
Epoch 5/5
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 28s 24ms/step - accuracy: 0.9199 - loss: 0.2448 - val_accuracy: 0.8643 - val_loss: 0.4105


In [29]:
# Step 8: Evaluate the Model
loss, accuracy = model.evaluate(X_test, y_test)

print("Test Loss:", loss)
print("Test Accuracy:", accuracy)



750/750 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.8595 - loss: 0.4226
Test Loss: 0.4225892722606659
Test Accuracy: 0.859541654586792


In [30]:
# predicting on news
news = ["India won the cricket world cup after defeating Australia."]

news = [clean_text(text) for text in news]

seq = tokenizer.texts_to_sequences(news)
pad = pad_sequences(seq, maxlen=max_length, padding='post')

prediction = model.predict(pad)

classes = ["World", "Sports", "Business", "Sci/Tech"]

print("Predicted Class:", classes[prediction.argmax()])


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 166ms/step
Predicted Class: Sports
